## 1. Import và tiện ích kiểm thử
Mã `diagonalize` nằm trong `Task1.py` (Python thuần, không NumPy). Notebook này chỉ import và định nghĩa các hàm nhỏ để nhân ma trận, norm Frobenius và sai số tương đối khi kiểm tra \(A \approx P D P^{-1}\).

Chạy notebook với thư mục làm việc là thư mục chứa `Task1.py` (ví dụ `projects`).

In [ ]:
import os
import sys
from pathlib import Path

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, "Task1.py")):
    if _cwd not in sys.path:
        sys.path.insert(0, _cwd)
else:
    raise ImportError(
        "Không thấy Task1.py trong thư mục làm việc hiện tại. "
        f"CWD={_cwd!r}. Mở đúng folder chứa Task1.py hoặc os.chdir(...)."
    )

from Task1 import diagonalize


def matmul(A, B):
    rows = len(A)
    cols = len(B[0])
    inner = len(B)
    result = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(rows):
        for j in range(cols):
            s = 0
            for k in range(inner):
                s += A[i][k] * B[k][j]
            result[i][j] = s
    return result


def frobenius_norm(M):
    s = 0.0
    for row in M:
        for x in row:
            s += abs(x) ** 2
    return s ** 0.5


def relative_error(A_ref, A_test):
    diff = []
    for i in range(len(A_ref)):
        row = []
        for j in range(len(A_ref[0])):
            row.append(A_test[i][j] - A_ref[i][j])
        diff.append(row)
    return frobenius_norm(diff) / max(frobenius_norm(A_ref), 1e-30)


def reconstruction_rel_err(A, tol=1e-10):
    """Trả về sai số tương đối Frobenius của A - P D P^{-1}; gây lỗi nếu vượt tol."""
    P, D, P_inv = diagonalize(A)
    recon = matmul(matmul(P, D), P_inv)
    err = relative_error(A, recon)
    if err >= tol:
        raise AssertionError(f"Sai số tương đối {err:.3e} >= {tol:.3e}")
    return err, P, D, P_inv

## 2. Kiểm thử tái tạo \(A = P D P^{-1}\) (5 trường hợp)
Ma trận tam giác trên tổng quát, đối xứng, đường chéo, phổ thực (2×2), và tam giác trên với trị riêng đã biết (3 và 7). Sai số tương đối cần \(< 10^{-10}\).

In [2]:
def test_diagonalize_reconstruction():
    print("--- Testing diagonalize reconstruction (A ~ P D P^-1) ---")

    # Case 1: 2×2 tam giác trên tổng quát
    A1 = [[4, 1], [0, 2]]
    e1, _, _, _ = reconstruction_rel_err(A1)
    print(f"1. Upper-triangular 2x2: rel_err={e1:.3e} (expected < 1e-10)")

    # Case 2: Đối xứng
    A2 = [[2, 1], [1, 2]]
    e2, _, _, _ = reconstruction_rel_err(A2)
    print(f"2. Symmetric: rel_err={e2:.3e} (expected < 1e-10)")

    # Case 3: Đường chéo — trị riêng 5 và 3
    A3 = [[5, 0], [0, 3]]
    e3, _, D3, _ = reconstruction_rel_err(A3)
    vals3 = sorted([D3[0][0], D3[1][1]])
    print(
        f"3. Diagonal: rel_err={e3:.3e}, eig(sorted)={vals3} (expected [3.0, 5.0])"
    )

    # Case 4: Phổ thực (2×2)
    A4 = [[3, 1], [0, 1]]
    e4, _, D4, _ = reconstruction_rel_err(A4)
    print(
        f"4. Real spectrum 2x2: rel_err={e4:.3e}, D[0][0]={D4[0][0]}, D[1][1]={D4[1][1]}"
    )

    # Case 5: Tam giác trên — trị riêng đường chéo 7 và 3
    A5 = [[7, 2], [0, 3]]
    e5, _, D5, _ = reconstruction_rel_err(A5)
    got5 = sorted([D5[0][0], D5[1][1]])
    print(f"5. Upper triangular: rel_err={e5:.3e}, eig(sorted)={got5} (expected [3.0, 7.0])")


test_diagonalize_reconstruction()

--- Testing diagonalize reconstruction (A ~ P D P^-1) ---
1. Upper-triangular 2x2: rel_err=0.000e+00 (expected < 1e-10)
2. Symmetric: rel_err=2.458e-16 (expected < 1e-10)
3. Diagonal: rel_err=0.000e+00, eig(sorted)=[3, 5] (expected [3.0, 5.0])
4. Real spectrum 2x2: rel_err=0.000e+00, D[0][0]=1, D[1][1]=3
5. Upper triangular: rel_err=0.000e+00, eig(sorted)=[3, 7] (expected [3.0, 7.0])


## 3. Kiểm thử xử lý lỗi (3 trường hợp)
Ma trận không vuông, không chéo hoá được (khối Jordan 2×2), và ma trận có trị riêng phức (xoay 90°) — cần báo lỗi rõ ràng.

In [5]:
def test_diagonalize_errors():
    print("--- Testing diagonalize error handling ---")

    # Case 1: Không vuông
    try:
        diagonalize([[1, 2, 3], [4, 5, 6]])
        print("1. Non-square: FAIL (expected ValueError)")
    except ValueError as ex:
        print(f"1. Non-square: OK — ValueError: {ex}")

    # Case 2: Không chéo hoá được (Jordan block)
    try:
        diagonalize([[1, 1], [0, 1]])
        print("2. Not diagonalizable: FAIL (expected ValueError)")
    except ValueError as ex:
        ok = "not diagonalizable" in str(ex).lower()
        tag = "OK" if ok else "OK (unexpected message)"
        print(f"2. Not diagonalizable: {tag} — {ex}")

    # Case 3: Trị riêng phức (2×2)
    try:
        diagonalize([[0, -1], [1, 0]])
        print("3. Complex eigenvalues: FAIL (expected ValueError)")
    except ValueError as ex:
        ok = "complex" in str(ex).lower()
        tag = "OK" if ok else "OK (unexpected message)"
        print(f"3. Complex eigenvalues: {tag} — {ex}")


test_diagonalize_errors()

--- Testing diagonalize error handling ---
1. Non-square: OK — ValueError: Matrix must be square
2. Not diagonalizable: OK — A is not diagonalizable
3. Complex eigenvalues: OK — Matrix has complex eigenvalues; cannot diagonalize over the real numbers
